# Exemples des nouveautés de la version 0.1

Ce notebook illustre les nouveautés de la version 0.1 : découpage automatique des emprises, requêtes parallèles, cache local, options par appel `paginate` et `output`. Il n'a pas encore été exécuté, l'API étant indisponible lors de sa rédaction : aucune sortie n'y figure.

## Configuration

Le jeton éventuel est lu dans la variable d'environnement `APIFONCIER_TOKEN`, jamais écrit en clair. Les requêtes d'un même appel s'exécutent en parallèle sur `MAX_WORKERS` threads.

In [ ]:
import os

import apifoncier

jeton = os.environ.get("APIFONCIER_TOKEN")  # nécessaire seulement pour Fichiers fonciers et DV3F
apifoncier.configure(MAX_WORKERS=4, PROGRESS_BAR=True)

## Accès libre et carte

Les friches de Cartofriches sont en accès libre. Les fonctions `geo*` renvoient un `GeoDataFrame`, que l'on représente directement.

In [ ]:
import apifoncier.cartofriches as cartofriches

gdf = cartofriches.geofriches(coddep="59")
ax = gdf.plot(column="site_type", legend=True, figsize=(8, 8))
ax.set_axis_off()

## Grande emprise découpée

L'API limite l'emprise d'une requête DVF+ à 0,02° de côté. L'emprise ci-dessous mesure 0,06° sur 0,04° : elle est découpée en six tuiles, chacune faisant l'objet d'une requête exécutée en parallèle. Les enregistrements strictement identiques renvoyés par plusieurs tuiles sont dédoublonnés. Le nombre de tuiles est plafonné par `MAX_TILES` (100 par défaut).

In [ ]:
import apifoncier.dvf_opendata as dvf

gdf = dvf.geomutations(in_bbox=[3.04, 50.62, 3.10, 50.66], anneemut=2022)
gdf.plot(markersize=2)

## Format et pagination par appel

`output` (`"pandas"`, `"polars"` ou `"dict"`) l'emporte sur `OUTPUT_FORMAT` pour l'appel considéré, et `paginate=False` ne récupère que la première page, soit `PAGE_SIZE` enregistrements. Le format polars exige l'extra `apifoncier[polars]`.

In [ ]:
df = dvf.mutations(code_insee="59350", anneemut=2022, output="polars")
apercu = dvf.mutations(code_insee="59350", paginate=False)

## Fonction générique

`apifoncier.get` interroge un endpoint de liste sans fonction dédiée. Elle accepte les mêmes options `paginate` et `output`.

In [ ]:
friches = apifoncier.get("/cartofriches/friches/", code_insee="59350")

## Cache local

Le cache, facultatif, exige l'extra `apifoncier[cache]`. Le second appel identique est servi depuis la base SQLite locale. Le jeton n'y est jamais écrit, mais les données à accès restreint y seraient conservées en clair : on vide le cache après usage.

In [ ]:
apifoncier.configure(CACHE=True)

df = dvf.mutations(code_insee="59350", anneemut=2022)  # interroge l'API
df = dvf.mutations(code_insee="59350", anneemut=2022)  # relu depuis le cache

apifoncier.clear_cache()